# cross-product-normal — faded example 1: Orient a triangle normal to face the camera

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `cross-product-normal`. Running the beacon reports progress on the `Geometry: Cross-product surface normal` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Geometry: Cross-product surface normal` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`cross-product-normal`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "cross-product-normal"
DD_SUBTOPIC = "Geometry: Cross-product surface normal"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

A cross-product normal `cross(P2-P1, P3-P1)` points along the winding-defined side of the face, which may point *away* from the viewer. To guarantee a front-facing normal, compute the unit normal, then flip its sign if its dot product with the view direction (centroid -> camera) is negative.

## Faded exercise 1

Implement `front_facing_normal(P1, P2, P3, cam)`. Compute the unit surface normal, then flip it so it points toward the camera. The triangle centroid is `(P1+P2+P3)/3`; the view direction is `cam - centroid`. Complete the missing line that produces the (possibly sign-flipped) outward unit normal.

**Fill in:** Flip the unit normal's sign when it points away from the camera (negative dot with the view direction).

In [ ]:
def front_facing_normal(P1: Tensor, P2: Tensor, P3: Tensor, cam: Tensor) -> Tensor:
    n = t.linalg.cross(P2 - P1, P3 - P1)
    normal = n / n.norm()
    centroid = (P1 + P2 + P3) / 3.0
    view = cam - centroid
    oriented = None  # TODO: Flip the unit normal's sign when it points away from the camera (negative dot with the view direction).
    return oriented


def _test():
    P1 = t.tensor([0.0, 0.0, 0.0])
    P2 = t.tensor([1.0, 0.0, 0.0])
    P3 = t.tensor([0.0, 1.0, 0.0])
    # camera below the plane (z < 0): raw normal is +z, should flip to -z
    cam = t.tensor([0.2, 0.2, -3.0])
    out = front_facing_normal(P1, P2, P3, cam)
    centroid = (P1 + P2 + P3) / 3.0
    view = cam - centroid
    assert out.shape == (3,)
    assert t.allclose(out.norm(), t.tensor(1.0), atol=1e-6)
    # must point toward the camera
    assert (out @ view).item() > 0
    # expected -z direction
    assert t.allclose(out, t.tensor([0.0, 0.0, -1.0]), atol=1e-6)
    # camera above the plane: keeps +z
    cam2 = t.tensor([0.2, 0.2, 3.0])
    out2 = front_facing_normal(P1, P2, P3, cam2)
    assert t.allclose(out2, t.tensor([0.0, 0.0, 1.0]), atol=1e-6)


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def front_facing_normal(P1: Tensor, P2: Tensor, P3: Tensor, cam: Tensor) -> Tensor:
    n = t.linalg.cross(P2 - P1, P3 - P1)
    normal = n / n.norm()
    centroid = (P1 + P2 + P3) / 3.0
    view = cam - centroid
    oriented = normal * t.sign(normal @ view)
    return oriented
```
</details>